In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

outcome_path = (
    ROOT / "02_intermediate/outcomes"
    / "primary_persistence_outcome_T1_T10_frozen.parquet"
)

backbone_path = (
    ROOT / "02_intermediate/integration"
    / "listing_origin_backbone_T1_T10.parquet"
)

for name, path in {
    "OUTCOME": outcome_path,
    "BACKBONE": backbone_path
}.items():
    print(f"\n===== {name} =====")
    print("Exists:", path.exists())

    if path.exists():
        df = pd.read_parquet(path)
        print("Shape:", df.shape)
        print("Columns:", df.columns.tolist())
        print(df.head(3).to_string(index=False))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

outcome = pd.read_parquet(
    ROOT / "02_intermediate/outcomes"
    / "primary_persistence_outcome_T1_T10_frozen.parquet"
)

backbone = pd.read_parquet(
    ROOT / "02_intermediate/integration"
    / "listing_origin_backbone_T1_T10.parquet"
)

# --------------------------------------------------
# STEP 1: Data integrity checks
# --------------------------------------------------

keys = ["transition", "listing_id"]

print("OUTCOME duplicates:", outcome.duplicated(keys).sum())
print("BACKBONE duplicates:", backbone.duplicated(keys).sum())

assert not outcome.duplicated(keys).any()
assert not backbone.duplicated(keys).any()

outcome_keys = pd.MultiIndex.from_frame(outcome[keys])
backbone_keys = pd.MultiIndex.from_frame(backbone[keys])

print("Keys match:", outcome_keys.equals(backbone_keys))

assert set(outcome_keys) == set(backbone_keys), (
    "OUTCOME and BACKBONE keys do not match."
)

# --------------------------------------------------
# STEP 2: Establish snapshot sequence
# --------------------------------------------------

snapshot_map = (
    outcome.groupby("transition")["snapshot_date"]
    .first()
    .pipe(pd.to_datetime)
    .sort_values()
)

transitions = snapshot_map.index.tolist()

print("\nSnapshot sequence:")
print(snapshot_map)

assert len(transitions) == 10
assert outcome["next_period_persistence"].isin([0, 1]).all()

# --------------------------------------------------
# STEP 3: Identify observed disappearance events
# --------------------------------------------------

events = outcome.loc[
    outcome["next_period_persistence"] == 0,
    ["transition", "listing_id", "snapshot_date", "outcome_date"]
].copy()

events["origin_index"] = events["transition"].map(
    {t: i for i, t in enumerate(transitions)}
)

# Listing presence at each observed origin snapshot
presence = {
    t: set(backbone.loc[
        backbone["transition"] == t, "listing_id"
    ])
    for t in transitions
}

# --------------------------------------------------
# STEP 4: Search for subsequent reappearance
# --------------------------------------------------

def find_reappearance(row):
    i = int(row["origin_index"])

    # Event is absence at snapshot i+1.
    # Reappearance can only be assessed from i+2 onward.
    if i + 2 >= len(transitions):
        return pd.Series({
            "has_followup": False,
            "reappeared": pd.NA,
            "first_reappearance_transition": pd.NA
        })

    listing_id = row["listing_id"]

    for j in range(i + 2, len(transitions)):
        t = transitions[j]

        if listing_id in presence[t]:
            return pd.Series({
                "has_followup": True,
                "reappeared": True,
                "first_reappearance_transition": t
            })

    return pd.Series({
        "has_followup": True,
        "reappeared": False,
        "first_reappearance_transition": pd.NA
    })

results = events.join(
    events.apply(find_reappearance, axis=1)
)

# --------------------------------------------------
# STEP 5: Summary by transition
# --------------------------------------------------

summary = (
    results.groupby("transition")
    .agg(
        disappearance_events=("listing_id", "size"),
        events_with_followup=("has_followup", "sum"),
        reappeared=("reappeared", lambda s: (s == True).sum())
    )
    .reset_index()
)

summary["reappearance_rate_pct"] = np.where(
    summary["events_with_followup"] > 0,
    100 * summary["reappeared"]
    / summary["events_with_followup"],
    np.nan
)

print("\n===== REAPPEARANCE SUMMARY =====")
print(summary.to_string(index=False))

# --------------------------------------------------
# STEP 6: Save audit outputs
# --------------------------------------------------

audit_dir = ROOT / "05_results/postfreeze_scientific_audit"
audit_dir.mkdir(parents=True, exist_ok=True)

results.to_csv(
    audit_dir / "A_reappearance_event_level.csv",
    index=False
)

summary.to_csv(
    audit_dir / "A_reappearance_summary_by_transition.csv",
    index=False
)

print("\nSaved to:", audit_dir)

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

AUDIT = ROOT / "05_results" / "postfreeze_scientific_audit"
AUDIT.mkdir(parents=True, exist_ok=True)

outcome = pd.read_parquet(
    ROOT / "02_intermediate/outcomes"
    / "primary_persistence_outcome_T1_T10_frozen.parquet"
)

backbone = pd.read_parquet(
    ROOT / "02_intermediate/integration"
    / "listing_origin_backbone_T1_T10.parquet"
)

keys = ["transition", "listing_id"]

# A. Key consistency
assert not outcome.duplicated(keys).any()
assert not backbone.duplicated(keys).any()

left_keys = pd.MultiIndex.from_frame(outcome[keys])
right_keys = pd.MultiIndex.from_frame(backbone[keys])

assert set(left_keys) == set(right_keys)
print("PASS: Outcome and backbone keys match as sets.")

# B. Snapshot sequence
dates = outcome[["transition", "snapshot_date"]].drop_duplicates()
dates["snapshot_date"] = pd.to_datetime(dates["snapshot_date"])

assert dates["transition"].is_unique

dates = dates.sort_values("snapshot_date")
transitions = dates["transition"].tolist()

presence = {
    t: set(
        backbone.loc[
            backbone["transition"] == t,
            "listing_id"
        ]
    )
    for t in transitions
}

# C. Verify next-snapshot labels
checks = []

for i, t in enumerate(transitions[:-1]):
    next_t = transitions[i + 1]

    sub = outcome[outcome["transition"] == t].copy()

    actual_next = sub["listing_id"].isin(
        presence[next_t]
    ).astype(int)

    mismatch = (
        actual_next.to_numpy()
        != sub["next_period_persistence"].to_numpy()
    ).sum()

    checks.append({
        "transition": t,
        "next_transition": next_t,
        "n": len(sub),
        "mismatches": int(mismatch)
    })

checks = pd.DataFrame(checks)

print("\n===== LABEL CONSISTENCY =====")
print(checks.to_string(index=False))

checks.to_csv(
    AUDIT / "A_label_consistency.csv",
    index=False
)

# Stop if label definitions do not agree
if checks["mismatches"].sum() > 0:
    raise ValueError(
        "Label mismatches detected. Investigate before continuing."
    )

# D. Reappearance at fixed T+2 horizon
records = []

for i, t in enumerate(transitions[:-2]):
    disappeared = outcome.loc[
        (outcome["transition"] == t)
        & (outcome["next_period_persistence"] == 0),
        ["listing_id"]
    ]

    target_t = transitions[i + 2]

    n = len(disappeared)

    reappeared = disappeared["listing_id"].isin(
        presence[target_t]
    ).sum()

    records.append({
        "transition": t,
        "origin_date": dates.iloc[i]["snapshot_date"],
        "missing_at": transitions[i + 1],
        "reappeared_at": target_t,
        "disappearance_events": n,
        "reappeared_T_plus_2": int(reappeared),
        "rate_T_plus_2_pct": (
            100 * reappeared / n if n else np.nan
        )
    })

fixed = pd.DataFrame(records)

print("\n===== FIXED T+2 REAPPEARANCE =====")
print(fixed.to_string(index=False))

fixed.to_csv(
    AUDIT / "A_reappearance_fixed_Tplus2.csv",
    index=False
)

# E. T3-specific investigation
t3 = fixed[fixed["transition"] == "T3"]

print("\n===== T3 INVESTIGATION =====")
print(t3.to_string(index=False))

print("\nSaved audit files to:")
print(AUDIT)

In [ ]:
from pathlib import Path

ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

for folder in [ROOT / "01_raw", ROOT / "02_intermediate"]:
    print(f"\n===== {folder.name} =====")

    if not folder.exists():
        print("Folder not found")
        continue

    matches = [
        p for p in folder.rglob("*")
        if p.is_file()
        and any(
            date in p.name
            for date in [
                "2026-08-13",
                "20260813",
                "2026_08_13"
            ]
        )
    ]

    for p in matches[:50]:
        print(p.relative_to(ROOT))

    print("Total matches:", len(matches))

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
AUDIT = ROOT / "05_results" / "postfreeze_scientific_audit"
AUDIT.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 1. Load frozen outcome and final snapshot
# --------------------------------------------------

outcome = pd.read_parquet(
    ROOT / "02_intermediate/outcomes"
    / "primary_persistence_outcome_T1_T10_frozen.parquet"
)

t11 = pd.read_parquet(
    ROOT / "02_intermediate/listings"
    / "listings_2026-08-13.parquet"
)

print("===== T11 SNAPSHOT =====")
print("Shape:", t11.shape)
print("Columns:", t11.columns.tolist())

assert "listing_id" in t11.columns, (
    "STOP: Check the listing ID column name."
)

assert t11["listing_id"].notna().all()
assert not t11["listing_id"].duplicated().any()

t11_ids = set(t11["listing_id"])

print("Unique T11 listing IDs:", len(t11_ids))

# --------------------------------------------------
# 2. Check frozen T10 labels
# --------------------------------------------------

t10 = outcome.loc[
    outcome["transition"] == "T10"
].copy()

actual_persistence = (
    t10["listing_id"].isin(t11_ids).astype(int)
)

mismatch = (
    actual_persistence.to_numpy()
    != t10["next_period_persistence"].to_numpy()
)

print("\n===== T10 LABEL CONSISTENCY =====")
print("T10 rows:", len(t10))
print("Mismatches:", int(mismatch.sum()))

t10_check = pd.DataFrame({
    "transition": "T10",
    "n": [len(t10)],
    "mismatches": [int(mismatch.sum())]
})

t10_check.to_csv(
    AUDIT / "A_T10_label_consistency.csv",
    index=False
)

if mismatch.sum() > 0:
    raise ValueError(
        "T10 label mismatches detected. "
        "Stop and investigate before proceeding."
    )

# --------------------------------------------------
# 3. T9 disappearance -> T11 reappearance
# --------------------------------------------------

t9_disappeared = outcome.loc[
    (outcome["transition"] == "T9")
    & (outcome["next_period_persistence"] == 0),
    ["listing_id"]
].copy()

t9_disappeared["reappeared_T_plus_2"] = (
    t9_disappeared["listing_id"].isin(t11_ids)
)

n = len(t9_disappeared)
r = int(t9_disappeared["reappeared_T_plus_2"].sum())

t9_summary = pd.DataFrame([{
    "transition": "T9",
    "disappearance_events": n,
    "reappeared_T_plus_2": r,
    "rate_T_plus_2_pct": (
        100 * r / n if n else np.nan
    )
}])

print("\n===== T9 FIXED T+2 REAPPEARANCE =====")
print(t9_summary.to_string(index=False))

t9_summary.to_csv(
    AUDIT / "A_T9_reappearance_fixed_Tplus2.csv",
    index=False
)

print("\nAudit files saved to:")
print(AUDIT)

In [ ]:
import pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
AUDIT = ROOT / "05_results" / "postfreeze_scientific_audit"

outcome = pd.read_parquet(
    ROOT / "02_intermediate/outcomes"
    / "primary_persistence_outcome_T1_T10_frozen.parquet"
)

backbone = pd.read_parquet(
    ROOT / "02_intermediate/integration"
    / "listing_origin_backbone_T1_T10.parquet"
)

t3 = outcome[outcome["transition"] == "T3"].copy()

t4_ids = set(
    backbone.loc[
        backbone["transition"] == "T4", "listing_id"
    ]
)

t5_ids = set(
    backbone.loc[
        backbone["transition"] == "T5", "listing_id"
    ]
)

# Verify T3 labels
assert (
    t3["listing_id"].isin(t4_ids).astype(int).to_numpy()
    == t3["next_period_persistence"].to_numpy()
).all()

t3["group"] = "persistent"

disappeared = t3["next_period_persistence"] == 0

t3.loc[
    disappeared & t3["listing_id"].isin(t5_ids),
    "group"
] = "temporary_absence"

t3.loc[
    disappeared & ~t3["listing_id"].isin(t5_ids),
    "group"
] = "continued_absence"

t3 = t3.merge(
    backbone.loc[
        backbone["transition"] == "T3",
        [
            "listing_id", "room_type", "property_type",
            "price", "number_of_reviews",
            "number_of_reviews_l30d", "last_scraped"
        ]
    ],
    on="listing_id",
    how="left",
    validate="one_to_one"
)

print("===== T3 GROUP COUNTS =====")
print(t3["group"].value_counts())

print("\n===== ROOM TYPE DISTRIBUTION =====")
print(
    pd.crosstab(
        t3["room_type"],
        t3["group"],
        normalize="columns"
    ).round(3)
)

print("\n===== GROUP DESCRIPTIVE STATISTICS =====")
print(
    t3.groupby("group")[
        ["price", "number_of_reviews", "number_of_reviews_l30d"]
    ].median()
)

t3.to_csv(
    AUDIT / "A_T3_disappearance_groups.csv",
    index=False
)

print("\nSaved:", AUDIT / "A_T3_disappearance_groups.csv")

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")
AUDIT = ROOT / "05_results/postfreeze_scientific_audit"
AUDIT.mkdir(parents=True, exist_ok=True)

backbone = pd.read_parquet(
    ROOT / "02_intermediate/integration"
    / "listing_origin_backbone_T1_T10.parquet"
)

# A. Verify prices
print("===== PRICE DIAGNOSTICS =====")

for col in ["price", "price_raw"]:
    s = backbone[col]
    print(f"\n{col}")
    print("dtype:", s.dtype)
    print("non-null:", s.notna().sum())
    print("sample values:", s.dropna().head(5).tolist())

price_numeric = pd.to_numeric(
    backbone["price"], errors="coerce"
)

print("\nNumeric price non-null:", price_numeric.notna().sum())
print("Numeric price median:", price_numeric.median())

# B. Check snapshot_date versus last_scraped
print("\n===== SNAPSHOT AVAILABILITY AUDIT =====")

dates = backbone[
    ["transition", "listing_id", "snapshot_date", "last_scraped"]
].copy()

dates["snapshot_date"] = pd.to_datetime(
    dates["snapshot_date"], errors="coerce"
)

dates["last_scraped"] = pd.to_datetime(
    dates["last_scraped"], errors="coerce"
)

dates["scrape_delay_days"] = (
    dates["last_scraped"] - dates["snapshot_date"]
).dt.total_seconds() / 86400

summary = (
    dates.groupby("transition")
    .agg(
        n=("listing_id", "size"),
        missing_snapshot_date=("snapshot_date",
                               lambda s: s.isna().sum()),
        missing_last_scraped=("last_scraped",
                              lambda s: s.isna().sum()),
        scraped_after_snapshot=("scrape_delay_days",
                                lambda s: (s > 0).sum()),
        scraped_on_snapshot=("scrape_delay_days",
                             lambda s: (s == 0).sum()),
        scraped_before_snapshot=("scrape_delay_days",
                                 lambda s: (s < 0).sum()),
        min_delay_days=("scrape_delay_days", "min"),
        median_delay_days=("scrape_delay_days", "median"),
        max_delay_days=("scrape_delay_days", "max")
    )
    .reset_index()
)

summary["pct_scraped_after"] = (
    100 * summary["scraped_after_snapshot"] / summary["n"]
)

print(summary.to_string(index=False))

print("\n===== DISTINCT DATE PAIRS =====")

pairs = (
    dates.groupby(
        ["transition", "snapshot_date", "last_scraped"],
        dropna=False
    )
    .size()
    .reset_index(name="listings")
    .sort_values(["transition", "listings"],
                 ascending=[True, False])
)

print(pairs.head(40).to_string(index=False))

summary.to_csv(
    AUDIT / "B_snapshot_availability_summary.csv",
    index=False
)

pairs.to_csv(
    AUDIT / "B_snapshot_last_scraped_pairs.csv",
    index=False
)

print("\nSaved audit tables to:", AUDIT)

In [ ]:
%run "C:/Users/yxt2919/7th_NZ_STR_Construction_AI/04_code/07_postfreeze_one_run_audit.py"

In [ ]:

# ============================================================
# PRICE DIAGNOSTICS / SNAPSHOT AVAILABILITY AUDIT
# Read-only scientific audit — no model retraining
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path(r"C:\Users\yxt2919\7th_NZ_STR_Construction_AI")

BACKBONE = (
    ROOT / "02_intermediate" / "integration"
    / "listing_origin_backbone_T1_T10.parquet"
)

assert BACKBONE.exists(), f"File not found: {BACKBONE}"

# Read only necessary columns
df = pd.read_parquet(
    BACKBONE,
    columns=[
        "transition",
        "listing_id",
        "snapshot_date",
        "last_scraped",
        "price",
        "price_raw"
    ]
)

print("=" * 85)
print("PART A — PRICE DIAGNOSTICS")
print("=" * 85)

print("\nTotal rows:", f"{len(df):,}")
print("Price dtype:", df["price"].dtype)
print("Price_raw dtype:", df["price_raw"].dtype)

# Numeric conversion without modifying original data
price_numeric = pd.to_numeric(
    df["price"], errors="coerce"
)

if price_numeric.notna().sum() == 0:
    price_numeric = pd.to_numeric(
        df["price_raw"].astype("string")
        .str.replace(r"[$,]", "", regex=True),
        errors="coerce"
    )

df["price_numeric"] = price_numeric

print("\nNonmissing price:", f"{price_numeric.notna().sum():,}")
print("Missing price:", f"{price_numeric.isna().sum():,}")
print("Missing rate:", f"{price_numeric.isna().mean():.2%}")
print("Median price:", price_numeric.median())

print("\nSample price values:")
print(df[["price", "price_raw", "price_numeric"]].head(10).to_string(index=False))

price_by_transition = (
    df.groupby("transition", sort=True)
    .agg(
        n=("listing_id", "size"),
        nonmissing_price=("price_numeric", "count"),
        median_price=("price_numeric", "median")
    )
    .reset_index()
)

price_by_transition["missing_price"] = (
    price_by_transition["n"]
    - price_by_transition["nonmissing_price"]
)

price_by_transition["missing_rate_pct"] = (
    100 * price_by_transition["missing_price"]
    / price_by_transition["n"]
)

print("\nPRICE BY TRANSITION:")
print(price_by_transition.to_string(index=False))

# ============================================================
# PART B — SNAPSHOT AVAILABILITY
# ============================================================

print("\n" + "=" * 85)
print("PART B — SNAPSHOT AVAILABILITY AUDIT")
print("=" * 85)

df["snapshot_date"] = pd.to_datetime(
    df["snapshot_date"], errors="coerce"
)

df["last_scraped"] = pd.to_datetime(
    df["last_scraped"], errors="coerce"
)

df["delay_days"] = (
    df["last_scraped"] - df["snapshot_date"]
).dt.total_seconds() / 86400

availability = (
    df.groupby("transition", sort=True)
    .agg(
        n=("listing_id", "size"),
        missing_scraped=("last_scraped", lambda x: x.isna().sum()),
        after_nominal=("delay_days", lambda x: (x > 0).sum()),
        on_nominal=("delay_days", lambda x: (x == 0).sum()),
        before_nominal=("delay_days", lambda x: (x < 0).sum()),
        median_delay_days=("delay_days", "median"),
        max_delay_days=("delay_days", "max")
    )
    .reset_index()
)

availability["pct_after_nominal"] = (
    100 * availability["after_nominal"] / availability["n"]
)

print("\nSNAPSHOT AVAILABILITY BY TRANSITION:")
print(availability.to_string(index=False))

print("\nOVERALL SUMMARY")
print("-" * 50)
print("Total observations:", f"{len(df):,}")
print("Scraped after nominal date:", f"{(df['delay_days'] > 0).sum():,}")
print("Scraped on nominal date:", f"{(df['delay_days'] == 0).sum():,}")
print("Scraped before nominal date:", f"{(df['delay_days'] < 0).sum():,}")
print("Missing scrape dates:", f"{df['last_scraped'].isna().sum():,}")

print("\n" + "=" * 85)
print("AUDIT COMPLETED — READ ONLY")
print("No frozen data, model, or predictions modified.")
print("=" * 85)


In [ ]:

%run "C:/Users/yxt2919/7th_NZ_STR_Construction_AI/04_code/08_finalize_publication_package.py"
